# AgentDojo: local models benchmark (Colab T4)

Tests whether prompt injection attacks work on three local Hugging Face models, using AgentDojo:

| Key | Model | Size |
|---|---|---|
| `qwen` | Qwen/Qwen3-8B-AWQ (4-bit) | ~6 GB |
| `llama` | hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 (4-bit) | ~6 GB |
| `nemotron` | nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16 | ~8 GB |

**How it works:** models are downloaded once to your Google Drive. Then, one model at a time, a vLLM
server loads the model on the T4 and AgentDojo runs 50 attacked tests plus 18 no-attack tests against it, 8 at a time.
All logs and results are saved to Drive.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. No Hugging Face token is needed (all three models are public).

**If Colab disconnects:** run cells 1–5 again, then cell 8. Finished tests are skipped automatically.

## 1. Settings
Change these if you want; everything below uses them.

In [ ]:
GITHUB_REPO_URL = "https://github.com/kaustubhadhe1206/AgentDojo_POC.git"
DRIVE_PROJECT_FOLDER = "/content/drive/MyDrive/agentdojo_local_benchmark"  # on your university Drive

MODELS_TO_TEST = ["qwen", "llama", "nemotron"]
ATTACKED_TESTS_PER_MODEL = 50   # spread over the workspace, slack and travel suites
PARALLEL_TESTS = 8              # tests sent to the model at the same time

# Derived paths (no need to change)
REPO_DIR = "/content/AgentDojo_POC"
MODELS_DIR = f"{DRIVE_PROJECT_FOLDER}/models"     # downloaded models (kept between sessions)
OUTPUT_DIR = f"{DRIVE_PROJECT_FOLDER}/output"     # logs and result CSVs (kept between sessions)
FAST_DISK_DIR = "/content/models"                 # local copy of the current model (fast to load)
VLLM_ENV = "/content/vllm-env"                    # separate Python environment for vLLM
MODELS_ARG = " ".join(MODELS_TO_TEST)

## 2. Check the GPU
You should see a **Tesla T4**. If you see an error, switch the runtime to T4 GPU.

In [ ]:
!nvidia-smi

## 3. Connect Google Drive
A sign-in popup appears; allow access with your university account.

> If this cell fails inside VS Code, open this notebook at [colab.research.google.com](https://colab.research.google.com) and run it there.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Drive folder ready:", DRIVE_PROJECT_FOLDER)

## 4. Get the code from GitHub
Clones the repository the first time, and pulls the latest version after that.

In [ ]:
import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 5. Install the libraries (about 5–10 minutes)
* AgentDojo goes into Colab's normal Python.
* vLLM goes into its own environment, because it brings its own versions of PyTorch and other
  libraries that could otherwise clash with AgentDojo or Colab.

In [ ]:
%pip install -q -r requirements.txt huggingface_hub

!pip install -q uv
!uv venv {VLLM_ENV} --python 3.12 -q
!uv pip install --python {VLLM_ENV}/bin/python -q "vllm>=0.15.1"
!{VLLM_ENV}/bin/vllm --version

## 6. Download the models to Drive (first time only, about 20 GB)
Skips models that are already downloaded, so it is safe to run again.

In [ ]:
!python phase6_local_models_benchmark.py --download-only --models {MODELS_ARG} --models-dir {MODELS_DIR}

## 7. Preview the tests
The same tests are used for every model, so the results are comparable.

In [ ]:
!python phase6_local_models_benchmark.py --plan-only --attacked-tests {ATTACKED_TESTS_PER_MODEL}

## 8. Run the benchmark (long: plan for several hours)
For each model: copy it from Drive to the local disk, start vLLM (a few minutes), run all tests,
stop vLLM, then move to the next model. One line is printed per finished test.

Results are saved after each model. If Colab disconnects, run cells 1–5 again and then this cell:
it continues where it stopped.

In [ ]:
!python phase6_local_models_benchmark.py \
    --models {MODELS_ARG} \
    --attacked-tests {ATTACKED_TESTS_PER_MODEL} \
    --workers {PARALLEL_TESTS} \
    --models-dir {MODELS_DIR} \
    --output-dir {OUTPUT_DIR} \
    --fast-disk-dir {FAST_DISK_DIR} --delete-fast-disk-copy \
    --vllm-command {VLLM_ENV}/bin/vllm

## 9. Results
* **baseline_utility**: tasks completed with no attack (can the model use the tools at all?)
* **utility_under_attack**: tasks still completed while under attack
* **attack_success_rate**: how often the model carried out the attacker's goal (lower is better)
* **ASR &lt;attack / suite&gt;**: the attack success rate broken down by attack style and by suite

Runs that crashed (e.g. the conversation got longer than the context window) are counted in `errors`
and left out of the percentages.

In [ ]:
import pandas as pd

summary = pd.read_csv(f"{OUTPUT_DIR}/results/model_summary.csv")
display(summary)

test_results = pd.read_csv(f"{OUTPUT_DIR}/results/test_results.csv")
attacked = test_results[test_results["attack"] != "none"].copy()
# True/False -> 1/0; crashed runs stay empty and are left out of the averages
attacked["attack_success"] = attacked["attack_success"].astype(str).map({"True": 1.0, "False": 0.0})
print("Attack success per model and attack style (1.0 = always succeeded):")
display(attacked.pivot_table(index="model", columns="attack", values="attack_success", aggfunc="mean").round(2))

## 10. Look at one conversation
Each test has a JSON log with the full conversation. Pick a row from `test_results` to read it.

In [ ]:
import json

row = attacked.iloc[0]   # change the number to look at another test
print(row[["model", "suite", "user_task", "injection_task", "attack", "utility", "attack_success"]], "\n")
log = json.load(open(row["log_file"]))
for message in log["messages"]:
    text = "".join(block["content"] or "" for block in (message["content"] or []))
    calls = [f'{c["function"]}({c["args"]})' for c in (message.get("tool_calls") or [])]
    print(f'--- {message["role"].upper()}\n{text[:1500]}', *calls, sep="\n")

## 11. If a model was SKIPPED
The vLLM server log shows why (for example, out of GPU memory). The other models still run.
Common fixes: lower `--max-model-len` (e.g. 16384) or `PARALLEL_TESTS` (e.g. 4) in cell 8.

In [ ]:
model_key = "nemotron"   # qwen, llama or nemotron
!tail -n 40 {OUTPUT_DIR}/server_logs/{model_key}.log